# Dashboard Interativo dos Campeões — Dissecação Final dos Vencedores PAC (Pós-Curadoria)

**Este notebook não é mais um passo prévio ao pipeline** (a antiga exploração visual "às cegas", anterior à triagem automática, foi aposentada). Ele é a camada final: um dashboard que consome diretamente `candidatos_vencedores_consolidados.csv` — os eventos de acoplamento teta→gama/HG que já passaram pelos portões:

- **Estatístico**: `veredito_refino == 'Candidato robusto'` (FDR + ajuste Gama + sem suspeita de banda larga).
- **FOOOF**: erro de ajuste < 0.15, knee válido, pico periódico real (sem contaminação 1/f) nas duas bandas.
- **Harmônico**: `veredito_harmonico == 'CLEAN'` (sem razão inteira suspeita com o teta).
- **Comportamental**: janela anotada, excluindo `Artefato / Cabo`.

**Uso:** escolha **Rato → Comportamento → Janela campeã** (ordenada por Z-score) nos controles abaixo e clique em **Visualizar** para gerar a dissecação em 4 painéis daquele evento específico:

1. **LFP bruto + banda teta (4–8 Hz) + banda gama (30–80 Hz)** — traçado temporal da janela de 10s.
2. **FOOOF banda baixa (2–45 Hz)** — fundo aperiódico (knee) e pico de teta destacado.
3. **FOOOF banda alta (35 Hz–~0,95×Nyquist)** — com limpeza de ruído de linha Kuhn (60 Hz) aplicada antes do ajuste, picos de gama/HG destacados.
4. **Comodulograma fase×amplitude** daquela janela de 10s específica.

*Nota de reprodutibilidade*: o comodulograma recalculado aqui usa surrogates com seed não fixada por padrão neste notebook exploratório, então o z-score de pico pode divergir um pouco do `z_score_refinado` gravado no CSV (mesma limitação já documentada em `comodulogram_interativo.py`). Os valores de FOOOF (cf_teta, cf_gama, R², knee), por serem determinísticos, devem bater com as colunas `*_fooof_v2`/`r2_*_fooof` do dataset mestre.


In [ ]:
# CELULA 1: IMPORTS CANONICOS E CONFIGURACAO
import os, sys, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# Localiza a raiz SCRIPT subindo os diretorios ate achar pac_core
cur = os.path.abspath(os.getcwd())
while cur and os.path.dirname(cur) != cur:
    if os.path.isdir(os.path.join(cur, 'pac_core')):
        break
    cur = os.path.dirname(cur)
SCRIPT_DIR = cur
if SCRIPT_DIR not in sys.path:
    sys.path.insert(0, SCRIPT_DIR)

# Nucleo compartilhado
from pac_core.io import fatia_janela, concatena_sessao
from pac_core.filtering import filtra_sinal, aplica_notch

# Nucleo do comodulograma
from pipeline.etapa3_comodulograma.comodulogram import (
    calcula_comodulograma_z, z_pico_par, bh_fdr_mapa, p_valores_por_celula,
    FASES_DEFAULT, AMPS_DEFAULT,
)

# Ajuste FOOOF + painel
from pipeline.etapa8_exploracao.comodulogram_interativo import (
    ajusta_fooof_teta_gamma, painel_fooof,
    FOOOF_TETA_FIT_RANGE, FOOOF_CTX_S,
    N_SURR, N_BINS, FDR_Q,
)

BASE_LAC_NOCI = os.path.join(os.path.dirname(SCRIPT_DIR), 'LAC_NOCI')
candidatos_csv = [
    os.path.join(SCRIPT_DIR, 'resultados', 'candidatos_vencedores_consolidados.csv'),
    os.path.join(SCRIPT_DIR, 'candidatos_vencedores_consolidados.csv'),
]
CSV_VENCEDORES = next((c for c in candidatos_csv if os.path.isfile(c)), candidatos_csv[0])

print('SCRIPT_DIR      :', SCRIPT_DIR)
print('BASE_LAC_NOCI   :', BASE_LAC_NOCI, '(existe:', os.path.isdir(BASE_LAC_NOCI), ')')
print('CSV_VENCEDORES  :', CSV_VENCEDORES, '(existe:', os.path.isfile(CSV_VENCEDORES), ')')


In [ ]:
# CELULA 2: CARREGA OS VENCEDORES CONSOLIDADOS E COLAPSA PSEUDOREPLICACAO ESPACIAL
df_vencedores = pd.read_csv(CSV_VENCEDORES)

# Convencao do projeto (ver CLAUDE.md #3): canais vizinhos com o mesmo pico
# na mesma janela sao UM UNICO evento biologico, nao descobertas independentes.
# Colapsa por (sessao, arquivo, janela_ini_s, janela_fim_s), mantendo a
# linha de maior z_score_refinado como representante do evento.
CHAVE_JANELA = ["sessao", "arquivo", "janela_ini_s", "janela_fim_s"]
df_eventos = (
    df_vencedores.sort_values("z_score_refinado", ascending=False)
    .drop_duplicates(CHAVE_JANELA)
    .reset_index(drop=True)
)

print(f"Linhas brutas (canal x par): {len(df_vencedores)}")
print(f"Eventos unicos (pos-colapso espacial): {len(df_eventos)}")
print()
print("Ratos:", sorted(df_eventos['rato'].dropna().unique()))
print("Comportamentos:", sorted(df_eventos['comportamento'].dropna().unique()))

df_eventos.head()


In [ ]:
# CELULA 3: RESOLUCAO DE CAMINHO + FIGURA DE DISSECACAO EM 4 PAINEIS

def resolve_pasta_basal(sessao_str, arquivo=None):
    """Localiza a pasta 'Basal antes da infusao' correspondente a um valor
    da coluna 'sessao' do dataset mestre.

    Busca dentro de LAC_NOCI/*/ (sem assumir de antemao se e grupo NOCI ou
    LAC) para nao depender de um mapa rato->pasta hardcoded, que quebraria
    silenciosamente se um novo rato/grupo for adicionado.

    ATENCAO: nomes de sessao como "Rodada-1-02-05-2024" NAO sao unicos --
    existem pastas com o MESMO nome em MTESC03_LAC e MTESC05_LAC (ratos
    diferentes rodados no mesmo protocolo/data). Se `arquivo` for
    informado e houver mais de um candidato, desempata escolhendo a pasta
    que realmente contem esse arquivo .ns2 -- sem isso, o glob pode
    silenciosamente resolver para o rato ERRADO.
    """
    nome_pasta = sessao_str
    sufixo = "_Basal antes da infusao"
    if nome_pasta.endswith(sufixo):
        nome_pasta = nome_pasta[: -len(sufixo)]

    candidatos = glob.glob(os.path.join(BASE_LAC_NOCI, "*", nome_pasta, "Basal antes da infusao"))
    if not candidatos:
        raise FileNotFoundError(
            f"Pasta 'Basal antes da infusao' nao encontrada para sessao={sessao_str!r} "
            f"(procurado: {nome_pasta!r} dentro de {BASE_LAC_NOCI})"
        )
    if len(candidatos) == 1 or arquivo is None:
        return candidatos[0]

    for c in candidatos:
        if os.path.isfile(os.path.join(c, arquivo)):
            return c
    raise FileNotFoundError(
        f"sessao={sessao_str!r} existe em {len(candidatos)} pastas "
        f"({candidatos}), mas nenhuma contem o arquivo {arquivo!r}"
    )


def plota_dissecacao_completa(row):
    """Gera a figura de 4 paineis (LFP, FOOOF teta, FOOOF gama/HG,
    comodulograma) para uma linha de df_eventos."""
    pasta_basal = resolve_pasta_basal(row["sessao"], row["arquivo"])
    dados, fs, ids_canais, offsets = concatena_sessao(pasta_basal)

    # canal na convencao do dataset mestre (1-based, ver processa_sessao.py
    # -- chan{indice_array+1}); ver correcao do off-by-one em comodulogram_interativo.py
    canal_idx = int(row["canal"]) - 1
    if not (0 <= canal_idx < dados.shape[1]):
        raise ValueError(f"Canal {row['canal']} (indice {canal_idx}) fora do range "
                         f"[0,{dados.shape[1]}) para {pasta_basal}")

    arquivos = sorted(f for f in os.listdir(pasta_basal)
                      if f.lower().endswith((".ns2", ".bin", ".dat")))
    idx_arquivo = arquivos.index(row["arquivo"])
    offset_s = offsets[idx_arquivo] / fs

    janela_ini_local = float(row["janela_ini_s"])
    janela_fim_local = float(row["janela_fim_s"])
    t_center = offset_s + (janela_ini_local + janela_fim_local) / 2.0
    par_ativo = row["par"]

    # --- janela de 10s: LFP bruto/teta/gama + comodulograma -----------------
    seg = fatia_janela(dados, fs, t_center - 5, t_center + 5)[:, canal_idx].astype(float)
    t_seg = np.arange(len(seg)) / fs

    # notch antes de filtrar p/ exibicao (regra do projeto: "Notch 60Hz em
    # tudo" -- sem isso o harmonico de 60Hz aparece como falsa oscilacao
    # gama na banda 30-80Hz)
    seg_limpo = aplica_notch(seg, fs, freqs_notch=[60, 120, 180, 240])
    seg_teta = filtra_sinal(seg_limpo, 4, 8, fs)
    seg_gama = filtra_sinal(seg_limpo, 30, 80, fs)

    # --- contexto de 45s p/ FOOOF (mesma janela usada por enriquece_dataset_mestre.py) --
    t_total_s = dados.shape[0] / fs
    ctx_ini = max(0.0, t_center - FOOOF_CTX_S / 2)
    ctx_fim = min(t_total_s, t_center + FOOOF_CTX_S / 2)
    sinal_ctx = fatia_janela(dados, fs, ctx_ini, ctx_fim)[:, canal_idx].astype(float)
    _, _, fm_teta, _, _, fm_gamma = ajusta_fooof_teta_gamma(sinal_ctx, fs)
    fit_range_gamma = (35.0, min(250.0, fs * 0.5 * 0.95))

    # --- comodulograma da janela de 10s -------------------------------------
    z_mapa, mi_obs, mi_surr = calcula_comodulograma_z(
        seg, fs, FASES_DEFAULT, AMPS_DEFAULT, n_surr=N_SURR, n_bins=N_BINS, retorna_mi=True)
    p_mapa = p_valores_por_celula(mi_obs, mi_surr)
    sig = bh_fdr_mapa(p_mapa, alpha=FDR_Q)
    z_mapa_plot = np.where(sig, z_mapa, np.nan)
    z_pico, fp_pico, fa_pico = z_pico_par(z_mapa, FASES_DEFAULT, AMPS_DEFAULT, par=par_ativo)
    pico_ok = z_pico is not None and not (isinstance(z_pico, float) and np.isnan(z_pico))

    # --- figura 4 paineis ----------------------------------------------------
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 11))
    fig.suptitle(
        f"{row['rato']} | {row['comportamento']} | canal {int(row['canal'])} | {par_ativo} | "
        f"z_dataset={row['z_score_refinado']:.2f} | {row['arquivo']} "
        f"[{janela_ini_local:.0f}-{janela_fim_local:.0f}s]",
        fontsize=12,
    )

    # painel 1: LFP bruto + teta + gama (empilhados verticalmente)
    off = 3.0 * np.std(seg_limpo)
    ax1.plot(t_seg, seg_limpo, color="#2c3e50", lw=0.8, label="Bruto (notch 60Hz)")
    ax1.plot(t_seg, seg_teta - off, color="#2980b9", lw=1.0, label="Teta 4-8Hz")
    ax1.plot(t_seg, seg_gama - 2 * off, color="#c0392b", lw=0.8, label="Gama 30-80Hz")
    ax1.set_title("LFP -- janela de 10s")
    ax1.set_xlabel("Tempo (s)")
    ax1.set_yticks([])
    ax1.legend(fontsize=8, loc="upper right")
    ax1.grid(alpha=0.2)

    # paineis 2/3: FOOOF teta e gama/HG (reusa logica ja validada, nao duplicada aqui)
    painel_fooof(ax2, fm_teta, cor_ap="#2980b9", cor_flat="#27ae60",
                cor_pico="#e74c3c", faixa_pico=(4.0, 12.0), rotulo="Teta",
                fit_range=FOOOF_TETA_FIT_RANGE)
    painel_fooof(ax3, fm_gamma, cor_ap="#8e44ad", cor_flat="#e67e22",
                cor_pico="#d35400", faixa_pico=fit_range_gamma, rotulo="Gama/HG",
                fit_range=fit_range_gamma)

    # painel 4: comodulograma
    im = ax4.pcolormesh(FASES_DEFAULT, AMPS_DEFAULT, z_mapa_plot, shading="auto", cmap="viridis")
    if pico_ok:
        ax4.scatter([fp_pico], [fa_pico], s=80, c="red", marker="X",
                   edgecolors="white", linewidths=1.5)
        titulo_comod = f"Comodulograma 10s (pico {par_ativo} z={z_pico:.2f})"
    else:
        titulo_comod = f"Comodulograma 10s (sem pico {par_ativo} significativo)"
    ax4.set_xlabel("Fase (Hz)")
    ax4.set_ylabel("Amplitude (Hz)")
    ax4.set_title(titulo_comod)
    fig.colorbar(im, ax=ax4, label="z")

    fig.tight_layout(rect=[0, 0, 1, 0.94])
    plt.show()
    return fig


In [ ]:
# CELULA 4: CONTROLES INTERATIVOS (Rato -> Comportamento -> Janela campeã)

def _comportamentos_do_rato(rato):
    sub = df_eventos[df_eventos["rato"] == rato]
    return sorted(sub["comportamento"].dropna().unique())


def _opcoes_janela(rato, comportamento):
    sub = df_eventos[(df_eventos["rato"] == rato) & (df_eventos["comportamento"] == comportamento)]
    sub = sub.sort_values("z_score_refinado", ascending=False)
    opcoes = []
    for idx, r in sub.iterrows():
        rotulo = (f"z={r['z_score_refinado']:.2f} | canal {int(r['canal'])} | {r['par']} | "
                 f"{r['arquivo']} [{r['janela_ini_s']:.0f}-{r['janela_fim_s']:.0f}s]")
        opcoes.append((rotulo, idx))
    return opcoes


ratos_disponiveis = sorted(df_eventos["rato"].dropna().unique())

rato_dd = widgets.Dropdown(options=ratos_disponiveis, description="Rato:", style={"description_width": "100px"})
comport_dd = widgets.Dropdown(description="Comportamento:", style={"description_width": "100px"})
janela_dd = widgets.Dropdown(description="Janela campeã:", style={"description_width": "100px"},
                             layout=widgets.Layout(width="500px"))
btn = widgets.Button(description="Visualizar", button_style="primary", icon="area-chart")
out = widgets.Output()


def _on_rato_change(change):
    opcoes = _comportamentos_do_rato(rato_dd.value)
    comport_dd.options = opcoes
    if opcoes:
        comport_dd.value = opcoes[0]


def _on_comport_change(change):
    janela_dd.options = _opcoes_janela(rato_dd.value, comport_dd.value)


def _on_click(_botao):
    with out:
        clear_output(wait=True)
        if janela_dd.value is None:
            print("Nenhuma janela disponível para esse filtro.")
            return
        row = df_eventos.loc[janela_dd.value]
        try:
            plota_dissecacao_completa(row)
        except Exception as e:
            print(f"Erro ao gerar visualização: {e}")


rato_dd.observe(_on_rato_change, names="value")
comport_dd.observe(_on_comport_change, names="value")
btn.on_click(_on_click)

_on_rato_change(None)
_on_comport_change(None)

display(widgets.VBox([
    widgets.HBox([rato_dd, comport_dd]),
    janela_dd,
    btn,
    out,
]))
